# 🌊 Lagoon Water Quality Monitor — Data Update

**How to use:** Click **Runtime → Run all**, authorise your Google account when prompted, wait for completion, then download the zip file.

Data is stored on Google Drive for persistent incremental updates. Each run only downloads new satellite imagery.

## 📦 Setup

In [ ]:
# Install dependencies
!pip install -q earthengine-api rasterio Pillow pandas

import ee
import json
import os
import time
import datetime
import requests
import numpy as np
import rasterio
import pandas as pd
from PIL import Image as PILImage
from pathlib import Path
from io import BytesIO
import shutil
import csv as csv_module

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
print("✓ Google Drive mounted")

# Authenticate GEE
ee.Authenticate(auth_mode="colab")
ee.Initialize(project="lagoon-dashboard-507406")
print("✓ GEE initialised")


## ⚙️ Configuration

In [ ]:
# ============================================================
# SETTINGS (do not change)
# ============================================================
GEE_ASSET = "projects/lagoon-dashboard-507406/assets/lagoon"
SATELLITE = "COPERNICUS/S2_SR_HARMONIZED"
CLOUD_MAX = 30
DATE_START = "2024-01-01"
SCALE = 10

FAI_VIS = {"min": -0.05, "max": 0.15, "palette": ["blue","cyan","green","yellow","orange","red"]}
NDCI_VIS = {"min": -0.1, "max": 0.5, "palette": ["blue","cyan","green","yellow","orange","red"]}

# Google Drive data folder (persistent storage)
DRIVE_DIR = Path("/content/drive/MyDrive/lagoon_dashboard_data")
FAI_DIR = DRIVE_DIR / "images" / "fai"
NDCI_DIR = DRIVE_DIR / "images" / "ndci"

# Create directories
FAI_DIR.mkdir(parents=True, exist_ok=True)
NDCI_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Drive data folder: {DRIVE_DIR}")


## 🗺️ Load Boundary & Detect Mode

In [ ]:
# Load lagoon boundary
lakes = ee.FeatureCollection(GEE_ASSET)
bounds = lakes.geometry().bounds()
bounds_coords = bounds.coordinates().getInfo()[0]
bounds_info = {"southwest": [bounds_coords[0][0], bounds_coords[0][1]], "northeast": [bounds_coords[2][0], bounds_coords[2][1]]}
print(f"✓ Lagoon boundary loaded")

# Detect existing data on Drive
existing_fai = sorted(FAI_DIR.glob("fai_*.png"))
existing_dates = [f.stem.replace("fai_", "") for f in existing_fai]

if existing_dates:
    last_date = max(existing_dates)
    from datetime import date, timedelta
    query_start = (date.fromisoformat(last_date) + timedelta(days=1)).isoformat()
    print(f"\nMode: INCREMENTAL")
    print(f"  Existing images on Drive: {len(existing_dates)}")
    print(f"  Latest: {last_date}")
    print(f"  Querying new data from: {query_start}")
else:
    query_start = DATE_START
    print(f"\nMode: FULL EXPORT (no existing data on Drive)")
    print(f"  Querying from: {query_start}")


## 🛰️ Query & Export New Images

In [ ]:
end_date = datetime.date.today().isoformat()

# Query Sentinel-2
s2 = (ee.ImageCollection(SATELLITE)
    .filterBounds(lakes)
    .filterDate(query_start, end_date)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_MAX))
    .sort("system:time_start")
    .map(lambda img: img.set("date", img.date().format("YYYY-MM-dd"))))

new_dates = s2.aggregate_array("date").distinct().sort().getInfo()
print(f"New images found: {len(new_dates)}")

if not new_dates:
    print("Already up to date. No new satellite imagery available.")
    print("(This may be due to cloud cover in recent acquisitions)")
else:
    for d in new_dates:
        print(f"  - {d}")


## 📊 Download New Images

In [ ]:
def compute_indices(image):
    red = image.select("B4").multiply(0.0001)
    nir = image.select("B8").multiply(0.0001)
    swir = image.select("B11").multiply(0.0001)
    fai = nir.subtract(red.add(swir.subtract(red).multiply((842-665)/(1610-665)))).rename("FAI")
    ndci = image.normalizedDifference(["B5","B4"]).rename("NDCI")
    return image.addBands(fai).addBands(ndci)

def download_image(image, index_name, vis_params, output_path):
    idx_image = image.select(index_name).clip(lakes)
    viz = idx_image.visualize(**vis_params)
    mask = ee.Image.constant(255).toByte().clip(lakes).unmask(0)
    rgba = viz.addBands(mask.rename("alpha"))
    url = rgba.getDownloadURL({"region": bounds, "scale": SCALE, "format": "GEO_TIFF", "bands": ["vis-red","vis-green","vis-blue","alpha"]})
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    with rasterio.open(BytesIO(resp.content)) as src:
        r,g,b,a = src.read(1), src.read(2), src.read(3), src.read(4)
        PILImage.fromarray(np.stack([r,g,b,a], axis=-1).astype(np.uint8), "RGBA").save(output_path, "PNG")

if new_dates:
    total = len(new_dates) * 2
    count = 0
    failed = []
    for date_str in new_dates:
        day_imgs = s2.filter(ee.Filter.eq("date", date_str))
        composite = day_imgs.map(compute_indices).mean()
        for idx, vis, out_dir, prefix in [("FAI",FAI_VIS,FAI_DIR,"fai"),("NDCI",NDCI_VIS,NDCI_DIR,"ndci")]:
            count += 1
            out_path = out_dir / f"{prefix}_{date_str}.png"
            if out_path.exists():
                print(f"  [{count}/{total}] {prefix}_{date_str}  ⊘ exists")
                continue
            for attempt in range(3):
                try:
                    download_image(composite, idx, vis, out_path)
                    print(f"  [{count}/{total}] {prefix}_{date_str}  ✓")
                    break
                except Exception as e:
                    if attempt < 2:
                        print(f"  [{count}/{total}] {prefix}_{date_str}  ✗ retry ({e})")
                        time.sleep(30)
                    else:
                        print(f"  [{count}/{total}] {prefix}_{date_str}  ✗ FAILED")
                        failed.append(f"{prefix}_{date_str}")
            time.sleep(2)
    if failed:
        print(f"\n⚠ {len(failed)} failed: {failed}")
    else:
        print(f"\n✓ All {total} images done")
else:
    print("No new images to download.")


## 📈 Generate Full Time Series CSV

Always regenerates from all available dates to ensure completeness.

In [ ]:
# Get ALL dates from Drive (existing + new)
all_dates = sorted([f.stem.replace("fai_","") for f in FAI_DIR.glob("fai_*.png")])
print(f"Total dates on Drive: {len(all_dates)}")

# Query full date range from GEE for statistics
s2_all = (ee.ImageCollection(SATELLITE)
    .filterBounds(lakes)
    .filterDate(DATE_START, datetime.date.today().isoformat())
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_MAX))
    .sort("system:time_start")
    .map(lambda img: img.set("date", img.date().format("YYYY-MM-dd"))))

print(f"Computing statistics for {len(all_dates)} dates...")
rows = []
for i, date_str in enumerate(all_dates):
    day_imgs = s2_all.filter(ee.Filter.eq("date", date_str))
    composite = day_imgs.map(compute_indices).mean()
    stats = composite.select(["FAI","NDCI"]).reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(),"",True),
        geometry=lakes.geometry(), scale=20, maxPixels=1e9
    ).getInfo()
    rows.append({
        "date": date_str,
        "FAI_mean": stats.get("FAI_mean"),
        "FAI_stdDev": stats.get("FAI_stdDev"),
        "NDCI_mean": stats.get("NDCI_mean"),
        "NDCI_stdDev": stats.get("NDCI_stdDev"),
    })
    if (i+1) % 10 == 0 or i == len(all_dates)-1:
        print(f"  {i+1}/{len(all_dates)} done")

df = pd.DataFrame(rows).drop_duplicates(subset="date",keep="last").sort_values("date")
df.to_csv(DRIVE_DIR / "full_timeseries.csv", index=False)
print(f"\n✓ CSV saved: {len(df)} rows")


## 📋 Generate Metadata & Dashboard Data

In [ ]:
# dates.json
with open(DRIVE_DIR / "dates.json", "w") as f:
    json.dump(all_dates, f)
print(f"✓ dates.json: {len(all_dates)} dates")

# bounds.json
with open(DRIVE_DIR / "bounds.json", "w") as f:
    json.dump(bounds_info, f, indent=2)
print("✓ bounds.json")

# dashboard_data.js
ts_data = []
with open(DRIVE_DIR / "full_timeseries.csv", "r") as f:
    for row in csv_module.DictReader(f):
        ts_data.append({
            "date": row["date"],
            "FAI_mean": float(row["FAI_mean"]) if row.get("FAI_mean") else None,
            "FAI_stdDev": float(row["FAI_stdDev"]) if row.get("FAI_stdDev") else None,
            "NDCI_mean": float(row["NDCI_mean"]) if row.get("NDCI_mean") else None,
            "NDCI_stdDev": float(row["NDCI_stdDev"]) if row.get("NDCI_stdDev") else None,
        })

js = f"""// Auto-generated - do not edit\nconst DATES = {json.dumps(all_dates)};\nconst BOUNDS = {json.dumps(bounds_info)};\nconst TIMESERIES = {json.dumps(ts_data)};\n"""
with open(DRIVE_DIR / "dashboard_data.js", "w") as f:
    f.write(js)
print(f"✓ dashboard_data.js: {len(ts_data)} data points")


## 📥 Download

Complete data package ready for download. Extract and replace your local `data/` folder.

In [ ]:
# Package Drive data into zip for download
temp_data = Path("/content/data")
if temp_data.exists():
    shutil.rmtree(temp_data)
shutil.copytree(DRIVE_DIR, temp_data)
zip_name = "/content/lagoon_dashboard_data"
shutil.make_archive(zip_name, "zip", "/content", "data")
zip_path = f"{zip_name}.zip"
size_mb = os.path.getsize(zip_path) / (1024*1024)

print(f"✓ Packaged: {size_mb:.1f} MB")
print(f"  Total dates: {len(all_dates)}")
print(f"  Latest: {all_dates[-1] if all_dates else 'N/A'}")

from google.colab import files
files.download(zip_path)
